# 3. Build an SVD from blocks and recipes

## What this notebook is about

Most SVD algorithms are a chain of a few standard steps. First reduce the matrix to a
simpler form (a triangular R from a QR factorization, or a bidiagonal B). Then solve the
small or simple problem. Then map the result back. Often you want to improve only one of
those steps, for example a QR that is faster on the GPU, and keep the rest as it is.

`svdlab` provides both levels:

- **blocks** (`svdlab.blocks`): the individual steps, such as `qr`, `bidiagonalize` and
  `bidiagonal_svd`. They are this repository's own readable implementations (tensor
  arithmetic only), so they run on CPU and GPU, real and complex.
- **recipes** (`svdlab.recipes`): complete SVDs built from blocks. Each step is a keyword
  argument, so `recipes.randomized(a, rank, qr=my_qr)` is randomized SVD with *your* QR.

```text
recipes.golub_kahan   a ──bidiagonalize──► Q, B, P ──bidiagonal_svd──► U = Q U_B, V = P V_B
recipes.qr_then_svd   a ──qr──► Q, R ──small_svd(R)──► U = Q U_R
recipes.randomized    a ──sketch──► A Ω ──qr──► Q ──(power iterations: qr)──► small_svd(Qᴴ A) ──► U = Q U_small
recipes.lanczos       a ──lanczos_bidiagonalize (k steps)──► U, B, V ──bidiagonal_svd──► U U_B, V V_B
recipes.jacobi        a ──jacobi_svd──► U, s, Vh
```

## What you will learn

1. What the main blocks compute, by calling them on a small matrix.
2. How to check the recipes as they are.
3. How to replace one step and see its trade-offs (CholeskyQR2, and your own QR).
4. How to find the slowest step by swapping in library versions.
5. How to tune a recipe's parameters, compose recipes, and use the Lanczos block.

## Words used here

| Word | Meaning |
| --- | --- |
| **QR factorization** | A = Q R, Q with orthonormal columns, R upper triangular |
| **bidiagonal matrix** | nonzero only on the diagonal `d` and the first superdiagonal `e`; every matrix can be reduced to one, A = Q B Pᴴ |
| **tall** | more rows than columns (m ≥ n). Some blocks accept only tall input; the recipes pass them the adjoint of a wide matrix |
| **sketch** | A Ω for a random Ω with a few columns: a cheap way to capture the dominant directions of A |
| **truncated case** | a case asking for fewer triplets than min(m, n). It is `ok` when the residual is within 1.05× the best possible one (`tail×`) |

About two minutes on a laptop CPU. Text version: [tutorial 3](../docs/tutorials/03-blocks-and-recipes.md).

## Setup

**What this cell does:** it finds the repository folder (so the notebook works from any
working directory), makes `svdlab` importable even if you have not installed it with pip,
and chooses where to compute: `DEVICE` is `'cuda'` when PyTorch sees a GPU and `'cpu'`
otherwise. To stay on the CPU even with a GPU present, set `USE_GPU = False`.

**What you should see:** one line with the PyTorch version and the chosen device. If you
get `ModuleNotFoundError`, the notebook kernel is not the Python environment where you ran
`pip install -e '.[laboratory]'`. Choose that environment as the kernel and run the cell
again.

In [1]:
# Setup: find the repository, choose the device. Run this cell first.
from pathlib import Path
import os, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'svdlab' / '__init__.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))      # works even if the package is not installed with pip
os.chdir(ROOT)                         # all paths below are relative to the repository root

import torch
import svdlab

USE_GPU = True                         # set to False to stay on the CPU even when a GPU is present
DEVICE = 'cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)               # same CPU thread count as the command line, comparable timings
print(f'torch {torch.__version__}, device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))

torch 2.14.1+cu130, device: cpu


## Step 1. Blocks, one at a time

**What this cell does:** it calls three blocks on a random 8×5 matrix and checks what they
promise:

- `blocks.qr(a)` → `Q, R` with `a = Q R`, R upper triangular;
- `blocks.bidiagonalize(a)` → `Q, d, e, P` with `a = Q B Pᴴ`, where B has diagonal `d` and
  superdiagonal `e`;
- `blocks.bidiagonal_svd(d, e)` → `U, s, V` with `B = U diag(s) Vᵀ`. Note that it returns V,
  not Vh. B has the same singular values as `a`.

**What you should see:** two reconstruction errors around 1e-15, `R upper: True`, and the
singular values from `bidiagonal_svd` equal to those from `torch.linalg.svdvals`.

In [2]:
from svdlab import blocks, recipes

a = torch.randn(8, 5, dtype=torch.float64, device=DEVICE)

q, r = blocks.qr(a)
print('QR:            |QR - A| =', float(torch.linalg.norm(q @ r - a)), ' R upper:', bool(torch.allclose(r, r.triu())))

q, d, e, p = blocks.bidiagonalize(a)          # A = Q B Pᴴ, B upper bidiagonal (d, e)
b = torch.diag(d) + torch.diag(e, 1)
print('bidiagonalize: |Q B Pᴴ - A| =', float(torch.linalg.norm(q @ b @ p.mH - a)))

ub, s, vb = blocks.bidiagonal_svd(d, e)       # B = Ub diag(s) Vbᵀ  (note: V, not Vh)
print('bidiagonal_svd: s =', s.cpu().numpy().round(4))
print('torch says:        ', torch.linalg.svdvals(a).cpu().numpy().round(4))

QR:            |QR - A| = 6.766044248900874e-16  R upper: True
bidiagonalize: |Q B Pᴴ - A| = 2.6240856461663545e-15
bidiagonal_svd: s = [3.068  2.6947 1.7128 1.3236 0.539 ]
torch says:         [3.068  2.6947 1.7128 1.3236 0.539 ]


**What this cell does:** it lists every block and recipe by name. The table in
[tutorial 3](../docs/tutorials/03-blocks-and-recipes.md#32-the-blocks) gives the inputs and
outputs of each block. In a notebook, `help(blocks.qr)` shows the same.

In [3]:
print('All blocks:', ', '.join(n for n in blocks.__all__ if n != 'ALGORITHMS'))
print('Recipes:   ', ', '.join(recipes.__all__))

All blocks: qr, torch_qr, bidiagonalize, lanczos_bidiagonalize, bidiagonal_svd, jacobi_svd, tridiagonalize, hermitian_eig, cholesky, solve_lower, solve_upper, random_sketch, complete_basis, svd, torch_svd
Recipes:    golub_kahan, qr_then_svd, randomized, lanczos, jacobi


## Step 2. The recipes as they are

Every recipe already has the `svd(a, rank)` signature, so it is a method and can be checked
directly.

**What this cell does:** it checks three recipes on the quick suite and shows the verdicts
as a table (`True` = ok).

**What you should see:** `True` everywhere. These are the unchanged algorithms you will
modify next.

In [4]:
rows = svdlab.compare({'golub_kahan': recipes.golub_kahan, 'qr_then_svd': recipes.qr_then_svd,
                       'randomized': recipes.randomized}, svdlab.QUICK_SUITE, device=DEVICE, verbose=False)
svdlab.table(rows).pivot(index='case', columns='method', values='ok')

method,golub_kahan,qr_then_svd,randomized
case,,,
clustered,True,True,True
complex_gaussian,True,True,True
gaussian_tall,True,True,True
gaussian_wide,True,True,True
ill_conditioned_1e8,True,True,True
low_rank_k4,True,True,True


## Step 3. Replace a step: CholeskyQR2 inside randomized SVD

CholeskyQR computes a QR factorization through the Gram matrix: YᴴY = L Lᴴ (a Cholesky
factorization), then Q = Y L⁻ᴴ. It is just matrix products and small triangular solves,
which GPUs do very fast. Doing it twice (CholeskyQR**2**) restores the orthogonality that
one pass loses. A replacement must keep the contract of the block it replaces, here
`blocks.qr`: tall `y (m, n)` in, `Q (m, n)` and upper triangular `R (n, n)` out.

**What this cell does:** it defines `cholesky_qr2`, builds randomized SVD with it
(`recipes.randomized(a, rank, qr=cholesky_qr2)`), and compares that with the unchanged
recipe. Besides the quick suite it adds the larger ill-conditioned matrix from the default
suite (160×120, condition number 1e8), where the difference shows.

**What you should see:** both versions `ok` on the quick matrices, CholeskyQR2 often
faster, and on `ill_conditioned_1e8_160x120` an `ERROR` for CholeskyQR2 saying the matrix
is not positive definite.

In [5]:
def cholesky_qr2(y):
    """Same contract as svdlab.blocks.qr: tall y (m, n) -> Q (m, n) orthonormal, R (n, n) upper."""
    q, r = y, torch.eye(y.shape[1], dtype=y.dtype, device=y.device)
    for _ in range(2):                            # the second pass repairs the lost orthogonality
        l = torch.linalg.cholesky(q.mH @ q)       # QᴴQ = L Lᴴ
        q = torch.linalg.solve_triangular(l.mH, q, upper=True, left=False)   # Q L⁻ᴴ
        r = l.mH @ r
    return q, r

def randomized_cholqr2(a, rank):
    return recipes.randomized(a, rank, qr=cholesky_qr2)

# The quick matrices plus the larger ill-conditioned one from the default set.
hard = [c for c in svdlab.DEFAULT_SUITE if c['name'] == 'ill_conditioned_1e8']
cases = list(svdlab.QUICK_SUITE) + [dict(hard[0], name='ill_conditioned_1e8_160x120')]
rows = svdlab.compare({'randomized': recipes.randomized, 'randomized_cholqr2': randomized_cholqr2},
                      cases, device=DEVICE, repeat=3)

randomized           gaussian_tall          ok     131.029 ms  recon 5.02e-15  tail× -  σ-rel 4.32e-15  orth 5.37e-15


randomized           gaussian_wide          ok      74.496 ms  recon 4.73e-15  tail× -  σ-rel 4.29e-15  orth 5.01e-15


randomized           ill_conditioned_1e8    ok     118.219 ms  recon 8.58e-15  tail× -  σ-rel 3.00e-10  orth 8.84e-15


randomized           clustered              ok     163.325 ms  recon 1.07e-14  tail× -  σ-rel 2.17e-14  orth 1.00e-14


randomized           complex_gaussian       ok      58.296 ms  recon 2.91e-15  tail× -  σ-rel 2.70e-15  orth 3.14e-15


...

randomized_cholqr2   ill_conditioned_1e8    ok      97.138 ms  recon 8.11e-15  tail× -  σ-rel 4.48e-10  orth 9.20e-15


randomized_cholqr2   clustered              ok      80.221 ms  recon 1.16e-14  tail× -  σ-rel 1.82e-14  orth 1.00e-14


randomized_cholqr2   complex_gaussian       ok      33.677 ms  recon 3.40e-15  tail× -  σ-rel 3.08e-15  orth 3.40e-15


randomized_cholqr2   low_rank_k4            ok      12.442 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 5.55e-16  orth 1.16e-15


randomized_cholqr2   ill_conditioned_1e8_160x120 ERROR _LinAlgError: linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 111 is not positive-definite).


**Why it fails:** YᴴY squares the condition number, the same effect as the Gram SVD in
notebook 2. With a condition number of 1e8 the Gram matrix has a condition number of 1e16,
so in float64 it is no longer positive definite, and the Cholesky factorization stops. The
failing case is reported as `ERROR` with the exception message, and the other cases still
run. The check makes this trade-off (faster, but not for ill-conditioned input) visible
instead of hiding it in an average.

The same experiment from a terminal, with the file
[examples/replace_a_step.py](../examples/replace_a_step.py):

```bash
python -m svdlab check examples/replace_a_step.py --compare svdlab.recipes:randomized
```

(`module:function` names any importable function as a method.)

## Step 4. Your turn: your own QR

**What this cell does:** it holds an example of a QR written from scratch for you to change:
classical Gram–Schmidt, applied twice per column (CGS2). For each column it removes the
components along the previous columns, twice for safety, and normalizes. It plugs this QR
into randomized SVD and checks it on the same cases.

**What you should see:** `ok` everywhere, including the ill-conditioned case, because the
second Gram–Schmidt pass removes what rounding left behind. Try deleting the second pass
(`range(1)`) and run again: orthogonality gets worse on the hard cases. It is also slow,
because of the Python loop over columns.

In [6]:
def my_qr(y):
    """YOUR CODE HERE. Classical Gram–Schmidt, twice per column (CGS2); same contract as blocks.qr."""
    m, n = y.shape
    q = torch.zeros_like(y)
    r = torch.zeros(n, n, dtype=y.dtype, device=y.device)
    for j in range(n):
        v = y[:, j]
        for _ in range(2):                        # the second pass removes what rounding left behind
            c = q[:, :j].mH @ v                   # components along the previous columns
            v = v - q[:, :j] @ c
            r[:j, j] += c
        r[j, j] = torch.linalg.vector_norm(v)
        q[:, j] = v / r[j, j]
    return q, r

rows = svdlab.check(lambda a, rank: recipes.randomized(a, rank, qr=my_qr), cases,
                    name='randomized_my_qr', device=DEVICE, repeat=1)

randomized_my_qr     gaussian_tall          ok      60.123 ms  recon 5.43e-15  tail× -  σ-rel 4.32e-15  orth 5.74e-15


randomized_my_qr     gaussian_wide          ok      58.139 ms  recon 4.81e-15  tail× -  σ-rel 3.83e-15  orth 4.94e-15


randomized_my_qr     ill_conditioned_1e8    ok      93.711 ms  recon 8.15e-15  tail× -  σ-rel 3.94e-10  orth 8.99e-15


randomized_my_qr     clustered              ok      97.249 ms  recon 1.11e-14  tail× -  σ-rel 1.20e-14  orth 1.07e-14


randomized_my_qr     complex_gaussian       ok      80.110 ms  recon 3.23e-15  tail× -  σ-rel 3.47e-15  orth 3.31e-15


randomized_my_qr     low_rank_k4            ok      27.082 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 4.78e-16  orth 9.83e-16


randomized_my_qr     ill_conditioned_1e8_160x120 ok    1859.271 ms  recon 4.77e-14  tail× -  σ-rel 4.44e-10  orth 5.19e-14


## Step 5. Find the slow step

The blocks are written to be readable and portable, not fast. To find out which step costs
the time, swap one block for the library version (`blocks.torch_qr`, `blocks.torch_svd`,
or `torch.linalg.svd` on B) and compare.

`functools.partial(recipe, step=...)` turns a recipe with one replaced step into an
`(a, rank)` function, without writing a wrapper function.

**What this cell does:** it times Golub–Kahan and QR-then-SVD as they are, and with library
versions of some steps, and prints the total time per variant.

**What you should see:** replacing only `bidiagonal_svd` makes Golub–Kahan about 50 times
faster. So almost all of its time is in the bidiagonal SVD (a Python loop of implicit-shift
QR steps here), and that is the step where a faster implementation would pay off.

In [7]:
import functools

def library_bidiagonal_svd(d, e):
    """Same contract as blocks.bidiagonal_svd: B = U diag(s) Vᵀ, returns (U, s, V)."""
    u, s, vh = torch.linalg.svd(torch.diag(d) + torch.diag(e, 1))
    return u, s, vh.mT

methods = {
    'golub_kahan': recipes.golub_kahan,
    'golub_kahan, library bidiagonal_svd': functools.partial(recipes.golub_kahan, bidiagonal_svd=library_bidiagonal_svd),
    'qr_then_svd': recipes.qr_then_svd,
    'qr_then_svd, library qr and small_svd': functools.partial(recipes.qr_then_svd, qr=blocks.torch_qr,
                                                               small_svd=blocks.torch_svd),
}
rows = svdlab.compare(methods, svdlab.QUICK_SUITE, device=DEVICE, repeat=3, verbose=False)
for s in svdlab.summary(rows):
    print(f"{s['method']:<40} {s['ok']}/{s['cases']} ok   total {s['total_median_ms']:9.1f} ms")

golub_kahan                              6/6 ok   total    3691.8 ms
golub_kahan, library bidiagonal_svd      6/6 ok   total      73.6 ms
qr_then_svd                              6/6 ok   total     735.7 ms
qr_then_svd, library qr and small_svd    6/6 ok   total       2.1 ms


## Step 6. Tune a parameter

`recipes.randomized` also takes numerical parameters:

- `oversample` (default 8): extra random columns beyond `rank`. More columns, better basis.
- `power_iterations` (default 2): how often to multiply by AᴴA to sharpen the basis.
  Important when the singular values decay slowly.
- `seed`: the random numbers used.

**What this cell does:** it runs three variants on two rank-10 problems. The first is a
matrix of exact rank 10 plus small noise (a clear gap after σ₁₀). The second has singular
values decaying slowly from 1 to 1e-3 (no gap). The table shows `tail×`, the residual
divided by the best possible rank-10 residual. A case passes when it is at most 1.05.

**What you should see:** without power iterations both problems miss the 1.05 limit. A lot
more oversampling helps the slow-decay case only. With the default two power iterations,
both are optimal (1.00).

In [8]:
cases = [
    dict(name='low_rank_k10', m=300, n=200, family='low_rank', rank=10, tail=1e-6, output_rank=10),
    dict(name='slow_decay', m=300, n=200, family='geometric', condition=1e3, output_rank=10),
]
variants = {
    'p0': functools.partial(recipes.randomized, power_iterations=0),
    'p0_oversample30': functools.partial(recipes.randomized, power_iterations=0, oversample=30),
    'p2 (default)': recipes.randomized,
}
rows = svdlab.compare(variants, cases, device=DEVICE, repeat=3, verbose=False)
svdlab.table(rows).pivot(index='method', columns='case', values='residual_over_optimal_tail').round(3)

case,low_rank_k10,slow_decay
method,,
p0,1.384,1.104
p0_oversample30,1.116,1.033
p2 (default),1.000,1.002


## Step 7. Lanczos: a step that only needs products with A

`blocks.lanczos_bidiagonalize(a, steps)` builds the bidiagonal matrix one column per step,
using only the products `a @ x` and `aᴴ @ y`. That makes it suitable for matrices you only
have as an operator (sparse, implicit). The largest singular values converge first, so a
few steps give the top of the spectrum. `recipes.lanczos` runs it and then
`bidiagonal_svd`, which we replace by the library version here, as in
[examples/lanczos_then_bdsqr.py](../examples/lanczos_then_bdsqr.py).

**What this cell does:** it runs Lanczos with 12 steps, the default (min(m, n, 2·rank + 8) =
28 here) and 60 steps on the same two rank-10 problems, and shows `tail×` and the largest
relative error of the top-10 singular values.

**What you should see:** with a clear gap (`low_rank_k10`) even 12 steps are optimal. For the
slowly decaying spectrum 12 steps fail (tail× above 1.05, values far off). 28 steps pass,
and 60 steps give the values to full precision. So the number of steps trades time for
accuracy, and how many you need depends on the spectrum.

In [9]:
lanczos_variants = {
    f'lanczos steps={k}': functools.partial(recipes.lanczos, steps=k, bidiagonal_svd=library_bidiagonal_svd)
    for k in (12, 28, 60)
}
rows = svdlab.compare(lanczos_variants, cases, device=DEVICE, repeat=3, verbose=False)
df = svdlab.table(rows)
df.pivot(index='method', columns='case', values=['residual_over_optimal_tail', 'resolved_spectrum_max_relative_error'])

residual_over_optimal_tail             \
case                           low_rank_k10 slow_decay   
method                                                   
lanczos steps=12                        1.0   1.126969   
lanczos steps=28                        1.0   1.000004   
lanczos steps=60                        1.0   1.000000   

                 resolved_spectrum_max_relative_error                
case                                     low_rank_k10    slow_decay  
method                                                               
lanczos steps=12                         1.577299e-15  6.069772e-01  
lanczos steps=28                         1.864081e-15  4.410502e-05  
lanczos steps=60                         1.864081e-15  3.219647e-15

## Step 8. Compose your own chain

A recipe is itself an SVD, so it can be a step of another recipe. And a new chain of blocks
is only a few lines.

**What this cell does:** it builds two equivalent methods. `qr_then_golub_kahan` is QR first,
then Golub–Kahan on the small R, composed with `functools.partial`. `qr_bidiag_chain` is the
same chain written out block by block, with the wide-matrix handling from notebook 2. Both
are checked on the quick suite.

**What you should see:** 6/6 ok for both.

In [10]:
qr_then_golub_kahan = functools.partial(recipes.qr_then_svd, small_svd=recipes.golub_kahan)

def qr_bidiag_chain(a, rank):
    """a = Q R, R = Q2 B Pᴴ, B = Ub diag(s) Vbᵀ, written out step by step."""
    wide = a.shape[0] < a.shape[1]
    x = a.mH if wide else a
    q, r = blocks.qr(x)
    q2, d, e, p = blocks.bidiagonalize(r)
    ub, s, vb = blocks.bidiagonal_svd(d, e)
    u = q @ q2 @ ub.to(x.dtype)                   # ub, vb are real; make them complex for complex input
    v = p @ vb.to(x.dtype)
    u, s, vh = u[:, :rank], s[:rank], v[:, :rank].mH
    return (vh.mH, s, u.mH) if wide else (u, s, vh)

rows = svdlab.compare({'qr_then_golub_kahan': qr_then_golub_kahan, 'qr_bidiag_chain': qr_bidiag_chain},
                      svdlab.QUICK_SUITE, device=DEVICE, repeat=1, verbose=False)
for s in svdlab.summary(rows):
    print(f"{s['method']:<22} {s['ok']}/{s['cases']} ok")

qr_then_golub_kahan    6/6 ok
qr_bidiag_chain        6/6 ok


## Summary

- Blocks are single steps with fixed contracts. Recipes are complete SVDs whose steps are
  keyword arguments.
- `functools.partial(recipe, step=my_step)` gives a checkable method with your step inside.
- Swapping in library blocks shows where the time goes.
- The check shows trade-offs per matrix: CholeskyQR2 is fast but breaks on
  ill-conditioned input; Lanczos and randomized SVD need enough steps or power iterations
  when the spectrum has no gap.

## Next

- [04_native_cpp_cuda.ipynb](04_native_cpp_cuda.ipynb): write a step or a whole SVD in C++ or CUDA.
- The 16 complete repository SVDs are available as builtin names (`'bidiag_qr'`,
  `'polar_qdwh'`, ...) and as `blocks.svd(a, algorithm)`.